# 📈 Model Evaluation & GradCAM Explainability
**RoadSense AI | Evaluation Notebook**

This notebook covers:
- Confusion matrix
- Per-class precision, recall, F1
- ROC-AUC curves
- GradCAM heatmap visualization
- Error analysis (misclassified samples)

In [ ]:
import os, sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import pickle, yaml
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

from src.data_preparation import load_dataset, create_data_generators
from src.gradcam import visualize_gradcam

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

model = tf.keras.models.load_model('../models/damage_classifier.h5')
with open('../models/class_names.pkl', 'rb') as f:
    cn_map = pickle.load(f)
class_names = [cn_map[i] for i in sorted(cn_map.keys())] if isinstance(cn_map, dict) else cn_map
print('Classes:', class_names)

## 1. Validation Predictions

In [ ]:
image_paths, labels = load_dataset(config['data']['dataset_path'])
_, val_gen = create_data_generators(
    image_paths, labels,
    batch_size=32, img_size=tuple(config['data']['image_size']),
    validation_split=0.2, seed=42
)
val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_gen.classes
print('Predictions done. Samples:', len(y_true))

## 2. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix.png', dpi=150)
plt.show()

## 3. Classification Report

In [ ]:
print(classification_report(y_true, y_pred, target_names=class_names))

## 4. ROC Curves

In [ ]:
n_classes = len(class_names)
y_true_bin = np.eye(n_classes)[y_true]
colors = ['#e94560', '#4a90d9', '#48bb78']

fig, ax = plt.subplots(figsize=(8, 6))
for i, (cls, col) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC={roc_auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves — One-vs-Rest')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/roc_curves.png', dpi=150)
plt.show()

## 5. GradCAM Visualization

In [ ]:
# Pick one sample per class for GradCAM
filenames = val_gen.filenames
for cls_id, cls_name in enumerate(class_names):
    idxs = [i for i, t in enumerate(y_true) if t == cls_id][:1]
    if not idxs: continue
    img_path = filenames[idxs[0]]
    print(f'GradCAM for {cls_name}: {img_path}')
    visualize_gradcam(
        model, class_names, img_path,
        img_size=tuple(config['data']['image_size']),
        save_path=f'../reports/figures/gradcam_{cls_name.lower()}.png'
    )

## 6. Error Analysis — Misclassified Samples

In [ ]:
wrong_idxs = [i for i in range(len(y_true)) if y_true[i] != y_pred[i]][:9]
print(f'Total misclassified: {len([i for i in range(len(y_true)) if y_true[i] != y_pred[i]])}')

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, idx in zip(axes.flat, wrong_idxs):
    from PIL import Image as PILImage
    img = PILImage.open(filenames[idx]).resize((224,224))
    ax.imshow(img)
    ax.set_title(f'True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}\nConf: {y_pred_probs[idx][y_pred[idx]]*100:.1f}%',
                 fontsize=9, color='red')
    ax.axis('off')
plt.suptitle('Misclassified Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/error_analysis.png', dpi=150)
plt.show()